# DLT Data process

We want to create a DLT pipeline using medaillon architecture to ingest daily stock data from Alpha Vantage API for four different symbols, create a dashboard using gold tables and orchestrate a daily workflow.

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType, IntegerType, DoubleType, LongType

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-7854847508576141>, line 1
----> 1 import dlt
      2 from pyspark.sql.functions import *
      3 from pyspark.sql.window import Window

File /databricks/python_shell/lib/dbruntime/autoreload/discoverability/hook.py:71, in AutoreloadDiscoverabilityHook._patched_import(self, name, *args, **kwargs)
     65 if not self._should_hint and (
     66     (module := sys.modules.get(absolute_name)) is not None and
     67     (fname := get_allowed_file_name_or_none(module)) is not None and
     68     (mtime := os.stat(fname).st_mtime) > self.last_mtime_by_modname.get(
     69         absolute_name, float("inf")) and not self._should_hint):
     70     self._should_hint = True
---> 71 module = self._original_builtins_import(name, *args, **kwargs)
     72 if (fname := fname or get_allowed_file_name_or_none(module)) is not None:
     7

### Get Bronze tables

In [0]:
@dp.view
def bronze_stock():
    return spark.read.table("jrvs_dlt.01_bronze.stock_data")

@dp.view
def bronze_company():
    return spark.read.table("jrvs_dlt.01_bronze.company_data")

@dp.view
def bronze_meta():
    return spark.read.table("jrvs_dlt.01_bronze.stock_meta_data")


### Bronze to Silver

In [0]:
@dp.table
@dp.expect("valid_volume", "volume >= 0")
@dp.expect("valid_close", "close >= 0")
@dp.expect("valid_date", "date <= current_date()")
def silver_stock():

    df = dp.read("bronze_stock")
    window = Window.partitionBy("symbol", "date").orderBy(col("ingestion_time").desc())

    return (
        df
        # Remove duplicate based on ingestion_time (we keep the most recent one)
        .withColumn("ingestion_rank", row_number().over(window))
        .filter("ingestion_rank = 1")
        .drop("ingestion_rank")

        # Remove NA
        .dropna(subset=['date', 'volume'])

        # Create new columns dates for aggregation
        .withColumn("date", to_date(col("date"), 'yyyy-MM-dd'))
        .withColumn("year", date_format(col("date"), "yyyy"))
        .withColumn("year_month", date_format(col("date"), "yyyy-MM"))
        .withColumn("week", weekofyear(col("date")))

        # Cast our columns
        .withColumn('open', col('open').cast(DecimalType(18,2)))
        .withColumn('high', col('high').cast(DecimalType(18,2)))
        .withColumn('low', col('low').cast(DecimalType(18,2)))
        .withColumn('close', col('close').cast(DecimalType(18,2)))

        # Create new columns change for informations
        .withColumn("change_day", col("close") - col("open"))
        .withColumn("change_day_pct", col("change_day") / col("open"))
        .withColumn("change_day", col("change_day").cast(DecimalType(18, 2)))
        .withColumn("change_day_pct", col("change_day_pct").cast(DecimalType(18, 4)))
    )


In [0]:
@dp.table
@dp.expect("valid_symbol", "symbol is not null")
def silver_meta():

    df = dp.read("bronze_meta")
    window = Window.partitionBy("symbol").orderBy(col("ingestion_time").desc())

    return (
        df
        # Remove duplicate based on ingestion_time (we keep the most recent one)
        .withColumn("ingestion_rank", row_number().over(window))
        .filter("ingestion_rank = 1")
        .drop("ingestion_rank")

        # Drop useless columns
        .drop("output_size")

        # Cast last_refreshed to date
        .withColumn('last_refreshed', to_date(col('last_refreshed'), 'yyyy-MM-dd'))
    )

In [0]:

@dp.table
@dp.expect("valid_symbol", "symbol is not null")
@dp.expect("valid_cik", "cik is not null")

def silver_company():

    df = dp.read("bronze_company")
    window = Window.partitionBy("symbol").orderBy(col("ingestion_time").desc())

    return (
        df
        # Remove duplicate based on ingestion_time (we keep the most recent one)
        .withColumn("ingestion_rank", row_number().over(window))
        .filter("ingestion_rank = 1")
        .drop("ingestion_rank")

        # Select only some colomns from the bronze table
        .selectExpr(
            "Symbol as symbol",
            "Name as name",
            "CIK as cik",
            "Country as country",
            "Sector as sector",
            "Industry as industry",
            "MarketCapitalization as market_cap",
            "SharesOutstanding as shares_outstanding",
            "Beta as beta",
            "PERatio as pe_ration",
            "ForwardPE as forward_pe",
            "PriceToBookRatio as price_to_book",
            "EVToEBITDA as ev_to_ebitda",
            "AnalystTargetPrice as analyst_target_price",
            "AnalystRatingStrongBuy as analyst_strong_buy",
            "AnalystRatingBuy as analyst_buy",
            "AnalystRatingHold as analyst_hold",
            "AnalystRatingSell as analyst_sell",
            "AnalystRatingStrongSell as analyst_strong_sell",
            "52WeekHigh as 52_week_high",
            "52WeekLow as 52_week_low"
        )

        # Cast our columns
        .withColumn('market_cap', col('market_cap').cast(LongType()))
        .withColumn('shares_outstanding', col('shares_outstanding').cast(LongType()))
        .withColumn('beta', col('beta').cast(DoubleType()))
        .withColumn('pe_ration', col('pe_ration').cast(DecimalType(18, 2)))
        .withColumn('forward_pe', col('forward_pe').cast(DecimalType(18, 2)))
        .withColumn('price_to_book', col('price_to_book').cast(DecimalType(18, 2)))
        .withColumn('ev_to_ebitda', col('ev_to_ebitda').cast(DecimalType(18, 2)))
        .withColumn('analyst_target_price', col('analyst_target_price').cast(DecimalType(18, 2)))
        .withColumn('analyst_strong_buy', col('analyst_strong_buy').cast(IntegerType()))
        .withColumn('analyst_buy', col('analyst_buy').cast(IntegerType()))
        .withColumn('analyst_hold', col('analyst_hold').cast(IntegerType()))
        .withColumn('analyst_sell', col('analyst_sell').cast(IntegerType()))
        .withColumn('analyst_strong_sell', col('analyst_strong_sell').cast(IntegerType()))
        .withColumn('52_week_high', col('52_week_high').cast(DecimalType(18, 2)))
        .withColumn('52_week_low', col('52_week_low').cast(DecimalType(18, 2)))
    )

### Silver to Gold


In [0]:
@dp.table
def gold_stock_daily():

    return (
        dp
        .read("silver_stock")
        .sort(col("symbol"), col("date").desc())
    )

In [0]:
@dp.table
def gold_stock_total_daily():

    return (
        dp
        .read("silver_stock")
        .groupBy("date")
        .agg(
            sum("volume").alias("volume"),
            sum("open").alias("open"),
            sum("high").alias("high"),
            sum("low").alias("low"),
            sum("close").alias("close")
        )
        .withColumn("change_day", (col("close") - col("open")).cast(DecimalType(18,2)))
        .withColumn("change_day_pct", (col("change_day") / col("open")).cast(DecimalType(18,4)))
        .sort(col("date").desc())
    )

In [0]:
@dp.table
def gold_stock_weekly():

    df = dp.read("silver_stock")
    window = Window.partitionBy("symbol", "year", "week").orderBy(col("date"))

    return (
        df
        # Add the open and close of week
        .withColumn("week_open", first("open").over(window))
        .withColumn("week_close", last("close").over(window))

        .groupBy("symbol", "year", "week")
        .agg(
            sum("volume").alias("volume"),
            first("week_open").alias("open"),
            max("high").alias("high"),
            min("low").alias("low"),
            last("week_close").alias("close")
        )
        .withColumn("change_week", (col("close") - col("open")).cast(DecimalType(18,2)))
        .withColumn("change_week_pct", (col("change_week") / col("open")).cast(DecimalType(18,4)))
        .sort(col("symbol"), col("year").desc(), col("week").desc())
    )


In [0]:
@dp.table
def gold_stock_monthly():

    df = dp.read("silver_stock")
    window = Window.partitionBy("symbol", "year_month").orderBy(col("date"))

    return (
        df
        # Add the open and close of week
        .withColumn("month_open", first("open").over(window))
        .withColumn("month_close", last("close").over(window))

        .groupBy("symbol", "year_month")
        .agg(
            sum("volume").alias("volume"),
            first("month_open").alias("open"),
            max("high").alias("high"),
            min("low").alias("low"),
            last("month_close").alias("close")
        )
        .withColumn("change_month", (col("close") - col("open")).cast(DecimalType(18,2)))
        .withColumn("change_month_pct", (col("change_month") / col("open")).cast(DecimalType(18,4)))
        .sort(col("symbol"), col("year_month").desc())
    )


In [0]:
@dp.table
def gold_stock_yearly():

    df = dp.read("silver_stock")
    window = Window.partitionBy("symbol", "year").orderBy(col("date"))

    return (
        df
        # Add the open and close of week
        .withColumn("year_open", first("open").over(window))
        .withColumn("year_close", last("close").over(window))

        .groupBy("symbol","year")
        .agg(
            sum("volume").alias("volume"),
            first("year_open").alias("open"),
            max("high").alias("high"),
            min("low").alias("low"),
            last("year_close").alias("close")
        )
        .withColumn("change_year", (col("close") - col("open")).cast(DecimalType(18,2)))
        .withColumn("change_year_pct", (col("change_year") / col("open")).cast(DecimalType(18,4)))
        .sort(col("symbol"), col("year").desc())
    )


In [0]:
@dp.table
def gold_meta_data_stock():
    return (
        dp
        .read("silver_meta")
    )

In [0]:
@dp.table
def gold_company():

    return (
        dp
        .read("silver_company")
        .selectExpr(
            "symbol",
            "name",
            "cik",
            "country",
            "sector",
            "industry",
            "market_cap",
            "shares_outstanding"
        )
    )

@dp.table
def gold_company_financials():

    return (
        dp
        .read("silver_company")
        .selectExpr(
            "symbol",
            "beta",
            "pe_ration",
            "forward_pe",
            "price_to_book",
            "ev_to_ebitda",
            "52_week_high",
            "52_week_low"
        )
    )

@dp.table
def gold_company_analyst():

    df = dp.read("silver_company")

    return (
        df
        .selectExpr(
            "symbol",
            "analyst_target_price",
            "analyst_strong_buy",
            "analyst_buy",
            "analyst_hold",
            "analyst_sell",
            "analyst_strong_sell",
        )
    )
